# 01 - YOLO11 Fine-tune

Notebook nay dung de fine-tune YOLO11 tren `cctv_person` va `custom_1/custom_2`.

Notebook da duoc chinh theo huong H100-aware de tan dung GPU RAM 96GB, system RAM lon, va I/O manh hon mac dinh.

Flow:
1. Setup Colab
2. Clone repo
3. Cai dependency
4. Tai `cctv_person` truc tiep va copy `custom_1/custom_2` tu Drive
5. Chay `pipelines/train_detector.py`
6. Kiem tra `best.pt`


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
REPO_URL = "https://github.com/ntdev204/adaptive-context-aware.git"
REPO_DIR = "/content/adaptive-context-aware"
CUSTOM1_DRIVE_DIR = "/content/drive/MyDrive/deep/data/fine_tuning/custom_1"
CUSTOM2_DRIVE_DIR = "/content/drive/MyDrive/deep/data/fine_tuning/custom_2"

In [ ]:
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -e .[engine,dev]
!pip install roboflow

In [ ]:
import os
import shutil

from roboflow import Roboflow

REPO_FINE_TUNING_DIR = os.path.join(REPO_DIR, "data", "fine_tuning")
CCTV_TARGET = os.path.join(REPO_FINE_TUNING_DIR, "cctv_person")
CUSTOM1_TARGET = os.path.join(REPO_FINE_TUNING_DIR, "custom_1")
CUSTOM2_TARGET = os.path.join(REPO_FINE_TUNING_DIR, "custom_2")

for path in (CCTV_TARGET, CUSTOM1_TARGET, CUSTOM2_TARGET):
    if os.path.exists(path):
        shutil.rmtree(path)
os.makedirs(REPO_FINE_TUNING_DIR, exist_ok=True)

rf = Roboflow(api_key="Ha4BZpmazCgHdw4IRPOA")
project = rf.workspace("car-vision").project("cctv-persons")
version = project.version(1)
dataset = version.download("yolov11")

downloaded_dir = dataset.location
shutil.move(downloaded_dir, CCTV_TARGET)
shutil.copytree(CUSTOM1_DRIVE_DIR, CUSTOM1_TARGET)
shutil.copytree(CUSTOM2_DRIVE_DIR, CUSTOM2_TARGET)
print("cctv_person ->", CCTV_TARGET)
print("custom_1 ->", CUSTOM1_TARGET)
print("custom_2 ->", CUSTOM2_TARGET)

In [ ]:
import os
import platform

import torch

print("python:", platform.python_version())
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("cpu_count:", os.cpu_count())

In [ ]:
DEVICE = 0
WORKERS = 8
PRETRAIN_EPOCHS = 50
FINETUNE_EPOCHS = 80
BATCH_SIZE = 64
IMGSZ = 960
BASE_MODEL = "yolo11m.pt"
ARTIFACT_DIR = "artifacts/detector_h100"

# Neu gap OOM, ha BATCH_SIZE ve 48 hoac 32.

In [ ]:
!python pipelines/train_detector.py \
  --base-model {BASE_MODEL} \
  --device {DEVICE} \
  --workers {WORKERS} \
  --work-dir {ARTIFACT_DIR} \
  --pretrain-epochs {PRETRAIN_EPOCHS} \
  --finetune-epochs {FINETUNE_EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --imgsz {IMGSZ}

In [ ]:
!ls -lah {ARTIFACT_DIR}/detector/finetune-school-multiclass/weights